# Universal GitHub Actions Control Center

Colab **Secrets** bölümüne yalnızca `GH_TOKEN` ekleyin ve notebook erişimini etkinleştirin. Arayüz Colab hücresinde yerel widget olarak açılır; parola veya ek secret gerekmez. Tek hücreyi çalıştırın.
Colab **Secrets** bölümünde yalnızca `GH_TOKEN` ekleyin ve notebook erişimini etkinleştirin. Tek hücreyi çalıştırın. Panel yerel `ipywidgets` ile Colab hücresinde görünür; Gradio sunucusu, iframe ve herkese açık bağlantı kullanılmaz.
**Güvenlik:** Panel yalnızca kendi Colab oturumunda çalışır. GitHub token'ını hücreye yazmayın; yalnızca Colab Secrets kullanın. Tüm run geçmişini silme ve Public yapma işlemleri yıkıcı/risklidir.


In [ ]:
#@title Universal GitHub Actions Control Center
#@markdown Add only GH_TOKEN to Colab Secrets, then run. The interface uses native Colab widgets; no public URL or extra login is created.
import sys,subprocess,importlib.util,re,json,time
def dep(m,p=None):
    if importlib.util.find_spec(m) is None: subprocess.check_call([sys.executable,"-m","pip","install","-q",p or m])
dep("requests"); dep("yaml","pyyaml")
import requests,yaml,base64,html,traceback
import ipywidgets as widgets
from IPython.display import display,HTML,clear_output
from google.colab import userdata
try: TOKEN=userdata.get("GH_TOKEN")
except Exception as e: raise RuntimeError("Colab Secrets bölümüne GH_TOKEN ekleyin ve notebook erişimini etkinleştirin.") from e
if not TOKEN: raise RuntimeError("Colab Secrets bölümünde GH_TOKEN boş olamaz.")
S=requests.Session(); S.headers.update({"Accept":"application/vnd.github+json","Authorization":"Bearer "+TOKEN,"X-GitHub-Api-Version":"2022-11-28"})
API="https://api.github.com"
def repo(x):
    x=x.strip().replace("https://github.com/","").strip("/")
    if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+",x): raise ValueError("Repo owner/name biçiminde olmalı.")
    return x
def req(m,p,**kw):
    z=S.request(m,API+p,timeout=30,**kw)
    if z.status_code>=400: raise RuntimeError(f"GitHub {z.status_code}: {z.text[:500]}")
    return {} if z.status_code==204 or not z.content else z.json()
def info(r): return req("GET",f"/repos/{repo(r)}")
def allruns(r):
    r=repo(r); out=[]; page=1
    while True:
        a=req("GET",f"/repos/{r}/actions/runs",params={"per_page":100,"page":page}).get("workflow_runs",[])
        out+=a
        if len(a)<100: break
        page+=1
    return out
def file_text(r,path):
    item=req("GET",f"/repos/{repo(r)}/contents/{path}",params={"ref":info(r)["default_branch"]})
    return base64.b64decode(item.get("content","")).decode("utf-8")
def discover(r):
    r=repo(r); ws=[]; benches=[]; presets=[]
    for w in req("GET",f"/repos/{r}/actions/workflows",params={"per_page":100}).get("workflows",[]):
        doc={}
        try: doc=yaml.safe_load(file_text(r,w["path"])) or {}
        except: pass
        trig=doc.get("on",doc.get(True,{})); d=trig.get("workflow_dispatch",{}) if isinstance(trig,dict) else {}
        ins=d.get("inputs",{}) if isinstance(d,dict) else {}
        ws.append((f"{w['name']} · {w['path']}",w["id"],bool(d),ins,w["path"]))
        if any(k in (w["name"]+" "+w["path"]).lower() for k in ("bench","benchmark","performance","profil")): benches.append(w["path"])
    try:
        data=json.loads(file_text(r,"CMakePresets.json"))
        presets=[x["name"] for x in data.get("buildPresets",[]) if x.get("name")]
    except: pass
    return ws,benches,presets
def table(r):
    return [[x.get("id"),x.get("name"),x.get("event"),x.get("head_branch"),x.get("status"),x.get("conclusion") or "—",x.get("created_at"),x.get("html_url")] for x in allruns(r)[:100]]
def inputs_for(r,label):
    x=next((x for x in discover(r)[0] if x[0]==label),None)
    return json.dumps({k:v.get("default","") for k,v in (x[3] or {}).items()},ensure_ascii=False,indent=2) if x else "{}"
def dispatch(r,ref,label,j,pub):
    r=repo(r); x=next((x for x in discover(r)[0] if x[0]==label),None)
    if not x or not x[2]: raise ValueError("Workflow dispatch desteklemiyor veya seçim geçersiz.")
    data=json.loads(j or "{}")
    if not isinstance(data,dict) or set(data)-set((x[3] or {}).keys()): raise ValueError("Input JSON nesnesi olmalı ve yalnızca tanımlı workflow inputlarını içermeli.")
    was_private=info(r)["private"]; changed=False
    try:
        if pub and was_private:
            req("PATCH",f"/repos/{r}",json={"private":False}); changed=True; time.sleep(4)
        started=time.time()
        req("POST",f"/repos/{r}/actions/workflows/{x[1]}/dispatches",json={"ref":ref.strip(),"inputs":data})
        if not (pub and was_private):
            return f"Dispatch gönderildi: {x[4]} / {ref}"
        # Keep the repository public until the dispatched run has completed.
        deadline=time.time()+3*60*60; run_id=None
        while time.time()<min(deadline,started+300) and run_id is None:
            batch=req("GET",f"/repos/{r}/actions/workflows/{x[1]}/runs",params={"event":"workflow_dispatch","branch":ref.strip(),"per_page":100}).get("workflow_runs",[])
            for candidate in batch:
                try:
                    created=candidate.get("created_at","").replace("Z","+00:00")
                    from datetime import datetime
                    if datetime.fromisoformat(created).timestamp() >= started-15:
                        run_id=candidate["id"]; break
                except: pass
            if run_id is None: time.sleep(8)
        if run_id is None: raise TimeoutError("Dispatch kabul edildi ancak 5 dakika içinde run bulunamadı; repository Private durumuna geri alınacak.")
        while time.time()<deadline:
            state=req("GET",f"/repos/{r}/actions/runs/{run_id}")
            if state.get("status")=="completed":
                return f"Run #{run_id} tamamlandı: {state.get('conclusion')}. Repository şimdi Private yapılacak."
            time.sleep(15)
        raise TimeoutError(f"Run #{run_id} üç saat içinde tamamlanmadı; repository Private yapılacak. Run GitHub Actions üzerinden izlenmeli.")
    finally:
        if changed:
            try: req("PATCH",f"/repos/{r}",json={"private":True})
            except Exception as e: raise RuntimeError("KRİTİK: Repository Private durumuna geri alınamadı. GitHub Settings > General > Danger Zone üzerinden hemen Private yapın. "+str(e))
def delete_ids(r,s):
    a=[x for x in re.split(r"[,\s]+",s or "") if x]
    if not a or len(a)>500 or any(not x.isdigit() for x in a): raise ValueError("1-500 arası sayısal run ID girin.")
    for x in a: req("DELETE",f"/repos/{repo(r)}/actions/runs/{x}")
    return f"{len(a)} run kaydı ve ilişkili log/artifact'lar silindi."
def delete_all(r,c):
    if c!="DELETE ALL RUN HISTORY": raise ValueError("Onay metni: DELETE ALL RUN HISTORY")
    a=allruns(r); done=0; errors=[]
    for x in a:
        try: req("DELETE",f"/repos/{repo(r)}/actions/runs/{x['id']}"); done+=1
        except Exception as e: errors.append(f"{x['id']}: {e}")
    return f"Silinen {done}/{len(a)} kayıt." + (" Hatalar: "+" | ".join(errors[:10]) if errors else "")
def visibility(r,v,c):
    if c!="CHANGE REPOSITORY VISIBILITY": raise ValueError("Onay metnini eksiksiz girin.")
    x=req("PATCH",f"/repos/{repo(r)}",json={"private":v=="Private"})
    return f"{x['full_name']} → {'Private' if x['private'] else 'Public'}"
# Native Colab widgets: no Gradio server, iframe, or public share URL.
repo_box=widgets.Text(value="Jue-dev/Tessera",description="Repository:",layout=widgets.Layout(width="48%"))
ref_box=widgets.Text(value="improve/stability-fuzzing-and-maintainability",description="Branch/ref:",layout=widgets.Layout(width="65%"))
connect_btn=widgets.Button(description="Connect & discover",button_style="primary",icon="plug")
refresh_btn=widgets.Button(description="Refresh run history",icon="refresh")
summary_out=widgets.Output()
workflow_dd=widgets.Dropdown(options=[],description="Workflow:",layout=widgets.Layout(width="95%"))
inputs_box=widgets.Textarea(value="{}",description="Inputs JSON:",layout=widgets.Layout(width="98%",height="130px"))
pub_check=widgets.Checkbox(value=False,description="Dispatch için Private → Public yap, sonra Private'a döndür (riskli)")
dispatch_btn=widgets.Button(description="Dispatch workflow",button_style="primary",icon="play")
dispatch_out=widgets.Output()
history_out=widgets.Output()
run_ids_box=widgets.Text(description="Run IDs:",placeholder="44716935108",layout=widgets.Layout(width="95%"))
delete_btn=widgets.Button(description="Seçili run kayıtlarını sil",button_style="danger",icon="trash")
delete_out=widgets.Output()
delete_all_box=widgets.Text(description='Onay:',placeholder="DELETE ALL RUN HISTORY",layout=widgets.Layout(width="95%"))
delete_all_btn=widgets.Button(description="Tüm Actions geçmişini sil",button_style="danger",icon="warning")
delete_all_out=widgets.Output()
visibility_dd=widgets.Dropdown(options=["Private","Public"],value="Private",description="Visibility:")
visibility_confirm=widgets.Text(description="Onay:",placeholder="CHANGE REPOSITORY VISIBILITY",layout=widgets.Layout(width="95%"))
visibility_btn=widgets.Button(description="Görünürlüğü değiştir",button_style="warning")
visibility_out=widgets.Output()
workflow_cache={}

def show_error(out):
    with out:
        traceback.print_exc()

def on_connect(_):
    with summary_out:
        clear_output(wait=True)
        try:
            r=repo(repo_box.value)
            i=info(r)
            ws,benches,presets=discover(r)
            workflow_cache.clear()
            workflow_cache.update({x[0]:x for x in ws})
            workflow_dd.options=[x[0] for x in ws]
            if ws:
                workflow_dd.value=ws[0][0]
                inputs_box.value=inputs_for(r,ws[0][0])
            print(f"Bağlandı: {i['full_name']} | {'Private' if i['private'] else 'Public'} | varsayılan dal: {i['default_branch']}")
            print(f"Workflow: {len(ws)} | Benchmark adayı: {len(benches)} | CMake build preset: {len(presets)}")
            print("\nWorkflow keşif sonuçları:")
            for x in ws:
                print(f" - {x[4]} | {'workflow_dispatch var' if x[2] else 'dispatch yok'}")
            print("\nBenchmark adayları:", ", ".join(benches) if benches else "Bulunamadı")
            print("CMake build presetleri:", ", ".join(presets) if presets else "Bulunamadı")
        except Exception:
            traceback.print_exc()

def on_workflow_change(change):
    if change.get("name")=="value" and change.get("new"):
        try:
            inputs_box.value=inputs_for(repo(repo_box.value),change["new"])
        except Exception as e:
            with summary_out:
                print("Input keşif hatası:",repr(e))
workflow_dd.observe(on_workflow_change,names="value")

def on_refresh(_):
    with history_out:
        clear_output(wait=True)
        try:
            rows=table(repo(repo_box.value))
            headers=["Run ID","Workflow","Event","Branch","Status","Conclusion","Created","URL"]
            print(f"Son {len(rows)} kayıt gösteriliyor (en fazla 100).")
            if rows:
                head="".join(f"<th>{html.escape(str(x))}</th>" for x in headers)
                body=""
                for row in rows:
                    cells=[]
                    for j,val in enumerate(row):
                        if j==7 and val:
                            cells.append(f'<td><a href="{html.escape(str(val),quote=True)}" target="_blank">GitHub run</a></td>')
                        else:
                            cells.append(f"<td>{html.escape(str(val if val is not None else ''))}</td>")
                    body+="<tr>"+"".join(cells)+"</tr>"
                display(HTML(f'<div style="overflow-x:auto"><table border="1" style="border-collapse:collapse;width:100%"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'))
            else:
                print("Run kaydı bulunamadı.")
        except Exception:
            traceback.print_exc()

def on_dispatch(_):
    with dispatch_out:
        clear_output(wait=True)
        try:
            print("Workflow dispatch isteği gönderiliyor...")
            result=dispatch(repo(repo_box.value),ref_box.value,workflow_dd.value,inputs_box.value,pub_check.value)
            print("SONUÇ:",result)
        except Exception:
            traceback.print_exc()

def on_delete(_):
    with delete_out:
        clear_output(wait=True)
        try:
            print(delete_ids(repo(repo_box.value),run_ids_box.value))
        except Exception:
            traceback.print_exc()

def on_delete_all(_):
    with delete_all_out:
        clear_output(wait=True)
        try:
            print("Tüm workflow run geçmişi siliniyor...")
            print(delete_all(repo(repo_box.value),delete_all_box.value))
        except Exception:
            traceback.print_exc()

def on_visibility(_):
    with visibility_out:
        clear_output(wait=True)
        try:
            print(visibility(repo(repo_box.value),visibility_dd.value,visibility_confirm.value))
        except Exception:
            traceback.print_exc()

connect_btn.on_click(on_connect)
refresh_btn.on_click(on_refresh)
dispatch_btn.on_click(on_dispatch)
delete_btn.on_click(on_delete)
delete_all_btn.on_click(on_delete_all)
visibility_btn.on_click(on_visibility)

workflow_panel=widgets.VBox([
    workflow_dd,inputs_box,pub_check,dispatch_btn,dispatch_out,
    widgets.HTML("<small>Benchmark iş akışları ad/path adına göre tahmin edilir. CMake build presetleri ayrıca listelenir. Çalıştırma, ilgili GitHub Actions workflow'u üzerinden yapılır.</small>")
])
history_panel=widgets.VBox([
    refresh_btn,history_out,run_ids_box,delete_btn,delete_out,
    widgets.HTML("<b style='color:#b91c1c'>DİKKAT:</b> Tüm geçmiş silme işlemi run kayıtlarını ve ilişkili log/artifact dosyalarını geri alınamaz biçimde siler."),
    delete_all_box,delete_all_btn,delete_all_out
])
security_panel=widgets.VBox([
    visibility_dd,visibility_confirm,visibility_btn,visibility_out,
    widgets.HTML("<small>Public yapmak kaynak kodunu ve commit geçmişini açığa çıkarır. Sonradan Private yapmak kopyaları/çatalları geri almaz.</small>")
])
tabs=widgets.Tab(children=[workflow_panel,history_panel,security_panel])
tabs.set_title(0,"Workflows & Benchmarks")
tabs.set_title(1,"Actions history")
tabs.set_title(2,"Repository security")
ui=widgets.VBox([
    widgets.HTML("<h2>Universal GitHub Actions Control Center</h2><p>Colab içi yerel panel · Public URL yok · Token arayüzde gösterilmez</p>"),
    widgets.HBox([repo_box,ref_box]),
    widgets.HBox([connect_btn,refresh_btn]),
    summary_out,
    tabs
])
display(ui)
print("Panel Colab hücresinde yerel ipywidgets olarak açıldı. Gradio/iframe/public share kullanılmıyor.")
